In [ ]:
import sys

if sys.version_info[0] + 0.1 * sys.version_info[1] < 3.6:
    raise Exception("Must be using Python 3.6+ version")

In [ ]:
%matplotlib inline
sys.path.append("..")

from plotly.subplots import make_subplots
import plotly.graph_objects as go
from wavelet_transform.postprocess.plotting import relabel
import numpy as np
import scipy.ndimage as ndi
 
import wavelet_transform.utils.wavelet_features as extract
from wavelet_transform.postprocess import plotting as plot
import wavelet_transform.utils.PCD_data as pcd
import wavelet_transform.wpt.inverse_transform as inv
from wavelet_transform.utils.extensions import fft, psd, to_dB, eng_form_plot, trim_data
from wavelet_transform.utils.machine_learning import fit_and_evaluate, fit_k_means, assess_parameter
import wavelet_transform.utils.kmeans_results as results
from wavelet_transform.utils.preprocessing import preprocess
from wavelet_transform.utils.save_plots import save_figure
from wavelet_transform.wpt.transform import Transform
import wavelet_transform.utils.statistical_features as sf

result_id is used to store the results in a file that can be found using results.get_data(result_id)

In [ ]:
saveflag = True

Font_Size = 24
Tick_Size = 24
Fig_Height = 400
Fig_Width = 800

In [ ]:
# PCD DATA
# "5-10", "5-34", "6-12" are the valid arguments for the 1st dataset
# tuples containing the voltage and frequency is used for the 2nd dataset
# e.g. (80, 1) for 80 mVpp, c1hat
frequency = 1.1 * 1e6
pcd_number = 1
voltage = 280
data, t_plot = pcd.get_data((voltage, pcd_number))  # [80, 120, 200, 280], [1, 5, 10]

no_clusters = 3, 6
result_id = f"{voltage}_{pcd_number}_{no_clusters[0]}_{no_clusters[1]}"

threshold = 3.25

In [ ]:
# from scipy.stats import cauchy, laplace, norm, chi2
# ch_pdf_data = cauchy.rvs(scale=1,size=100000)
# lap_pdf_data = laplace.rvs(size=100000)
# gaus_pdf_data = norm.rvs(scale=1,size=1000000)
# chis_pdf_data = chi2.rvs(df=2,size=100000)

# data = gaus_pdf_data
# t_plot = np.linspace(0,1,1000000)

In [ ]:
# # data from 16/11/23
# frequency = 2e6
# exp_number = 36  # currently testing 9 (350 mVpp), 15 (10? mVpp), 55 (350 mVpp transient)
# data, t_plot= pcd.get_data(exp_number)
# result_id = str(exp_number)

- test 2MHz with 400, 450 amplitude and high wave periods per pulse for interesting signals
- write up numerical studies and ga data plots

In [ ]:
sampling_rate = np.round(1 / (t_plot[1] - t_plot[0]))
freq_threshold = 10e6 / (sampling_rate / 2)
fft_lim = 10  # / sampling_rate / 2
# frequency = fft_lim

In [ ]:
# Parameters for the wavelet transform
wavelet = "db30"
levels = None  # set to None to use max-level transform
mode = "symmetric"
best_basis = "min_entropy"
min_level = 7
# keep_ratio = "valid"
inv_keep_ratio = .9

In [ ]:
# parameters = {
#     "amplitude": 200,
#     "frequency": "2",
#     "duty cycle": 10,
#     "prf": 20,  # pulse repetition frequency
#     "experiment": "00002",
# }

# if parameters["frequency"]=="2":
#     frequency = 2e6 #parameters["frequency"] * 1e6
# else:
#     frequency = 7.37e6
    
# dfbs = pcd.get_data(parameters)
# data, t_plot = dfbs["c2"]
# excitation_data, t_plot_ext = dfbs["c1"]

# result_id = (parameters["amplitude"], parameters["frequency"])

In [ ]:
# k-means parameters
kmeans_params = {
    "init": "k-means++",
    "n_init":10,
    "max_iter":100,
    "tol":1e-8,
    "verbose":0,
    "algorithm":"lloyd",
}

# training data
classifying_metrics = [
    "log average energy",
    "kurtosis",
    # "crest factor",
    # "level entropy",
    "spectral entropy",
    "average energy",
    # "skewness"
]
# Set to no_clusters to None to select optimal number of clusters by silhouette score
use_inverse_metrics = True

In [ ]:
# indx = np.abs(excitation_data) > 0.001
# indx = np.where(indx != 0)[0][0]
# pulse_rep_length = int(sampling_rate * (1 / (parameters["prf"] * 1e3)))
# indx_end = indx + pulse_rep_length
# data_segments = []
# excitation_data_segments = []

# while indx_end < len(data):
#     data_segments.append(data[indx:indx_end])
#     excitation_data_segments.append(excitation_data[indx:indx_end])
#     indx = indx_end
#     indx_end = indx_end + pulse_rep_length

In [ ]:
# data_segments[0].shape

In [ ]:
# %matplotlib ipympl
# import matplotlib.pyplot as plt
# plt.plot([np.linspace(1,len(data_segments[0]),len(data_segments[0])) for i in range(len(data_segments))],np.vstack(data_segments),'-')
# plt.show()

In [ ]:
# Plot PCD data and the excitation sound wave
# fig = make_subplots(rows=1, cols=1)
# fig.add_trace( go.Scatter(x=t_plot, y=data, mode='lines',name='data'),
#               row=1,col=1)
# # fig.add_trace( go.Scatter(x=t_plot, y=excitation_data ,mode='lines', name = 'excitation data'),
# #               row=1,col=1)

# fig.update_layout(height=600, width=1000,
#                   xaxis1 = dict(title='Time [s]'),
#                   yaxis1 = dict(title='Magnitude [DL]'),)
# fig.show()

In [ ]:
# Trim the array down to a start and end time
t_lims = (20e-6, 60e-6)
data_trim, t_plot = trim_data(data, t_plot, t_lims, return_time=True)
data_processed = preprocess(data_trim, window=False)

d1 0.0040870003629810215

d2 0.05769716101573163

In [ ]:
# Plot generated data
# fft_data, freq = fft(data_processed, sampling_rate, frequency)
def plot_psd(data, t):
    psd_data, freq = psd(data, sampling_rate, frequency)

    t_plot = [i*10**6 for i in t]
    # plot sound wave and fft
    fig = make_subplots(rows=2, cols=1)
    fig.add_trace(
        go.Scatter(x=t_plot, y=data, mode="lines", name="data"), row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=freq, y=to_dB(psd_data), mode="lines", name="fft - amp"),
        row=2,
        col=1,
    )
    # fig.add_trace( go.Scatter(x=freq, y=np.angle(fft_data, deg=True),mode='lines', name = 'fft - phase'),
    #               row=3,col=1)
    
    fig.update_layout(
        height=Fig_Height*2,
        width=Fig_Width,
        font_size=Font_Size,
        xaxis1=dict(title="Time [&mu;s]"),
        xaxis2=dict(title="Freq [f/f<sub>0</sub>]", range=[0, fft_lim]),
        xaxis3=dict(title="Freq [f/f<sub>0</sub>]", range=[0, fft_lim]),
        yaxis1=dict(title="Magnitude [unit]"),
        yaxis2=dict(title="Magnitude [dB]", range=[-50, 50]), 
        yaxis3=dict(title="Magnitude [rad]"), 
        # legend=dict(x=0.8, y=1, font=dict(size=Font_Size)),
        plot_bgcolor="white",     
    )
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgrey')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgrey')        
    fig.show()
    return fig

In [ ]:
fig = plot_psd(data_processed, t_plot)
filename = f"{result_id}_data"
print(filename)
if saveflag is True:
    save_figure(fig, filename)

## Wavelet transform

In [ ]:
wpt_df, tr_obj = extract.extract_wavelet_features(
    data_processed, levels, wavelet, mode, 
    include_tr_obj=True, best_basis=best_basis,
    min_level=min_level, freq_threshold=freq_threshold,
    # keep_ratio=keep_ratio
)

if use_inverse_metrics is True:
    inv_tr_obj = inv.InverseTransform(tr_obj)  # Use info from inverse transform object instead
    wpt_df_train = inv_tr_obj.info(
        leaf_nodes=True, freq_threshold=freq_threshold, keep_ratio=inv_keep_ratio
    )
else:
    wpt_df_train = wpt_df.copy()

## Training

In [ ]:
data_train, labels, data_train_normalised, centers, scaler, kmeans = fit_k_means(
    wpt_df_train, 
    classifying_metrics, 
    no_clusters=no_clusters, 
    kmeans_params=kmeans_params, 
)
centers_scaled = scaler.fit_transform(centers)
centers_sorted, labels_sorted = relabel(labels, centers, "average energy", ascending=False)

In [ ]:
# fit_and_evaluate(kmeans, data_train_normalised, labels, name='wpt data', n_runs=100)

In [ ]:
# parameter_of_interest = "max_iter"
# values = [i for i in range(1, 100, 2)]
# rand_indices = assess_parameter(
#     parameter_of_interest, values, wpt_df_train, 
#     classifying_metrics, no_clusters
# )

# plt.plot(values, rand_indices)
# plt.title(f"Effect of {parameter_of_interest} on clustering algorithm")
# plt.ylabel("Adjusted rand index")
# plt.xlabel(f"{parameter_of_interest.capitalize()}")
# plt.show()

## Save results

## Plot results

In [ ]:
fig = plot.plot_clustering(
    data_train, 
    centers_sorted, 
    labels_sorted, 
    [classifying_metrics[i] for i in [0, 1, 2]], 
    title=None
)

In [ ]:
# Plot centroid metrics
xaxis_ticks_labels = dict((i, f"C{i}") for i in range(len(centers)))
fig = plot.plot_centers(classifying_metrics, centers_sorted, label_map=xaxis_ticks_labels)
print(centers_sorted)

In [ ]:
tr_obj.calc_frequencies(sampling_rate)
fig = plot.heisenberg(
    tr_obj, t_plot[-1], max_freq=fft_lim*1e6/(sampling_rate/2), padding="repeat", use_log_scale=True,
    vmin=-2, vmax=2, color_map="viridis",
)


In [ ]:
if tr_obj.max_level() < 7:
    tr_obj.plot_tree(
        node_size="average energy",
        node_colour=list(labels_sorted), 
        freq_threshold=freq_threshold, 
        order="natural",
        sampling_rate=sampling_rate,
        show_frequency=False,
        fig_name= None,  #'dwpt_graph_' + result_id,
    )

In [ ]:
# Set clusters to None to plot all of them, otherwise, plot a list of int
fig = plot.plot_fft_by_group(
    tr_obj, labels_sorted, sampling_rate,
    clusters=None, 
    xlims=[0,fft_lim], 
    ylims=[-100,50],
    fundamental_freq=frequency,
    freq_threshold=freq_threshold, 
    fig_height=Fig_Height, 
    fig_width=Fig_Width,
    plot_full_reconstruction=False  
)

fig = plot.plot_reconstructions_by_group(
    tr_obj, labels_sorted, t_plot, freq_threshold,
    fig_height=Fig_Height, 
    fig_width=Fig_Width,
)

In [ ]:
import wavelet_transform.wpt.inverse_transform as inv

def trim(data, ratio):
    lim = int((1 - ratio) / 2 * len(data))
    if lim == 0:
        return data
    return data[lim:-lim]

# Add number to plot_centres to plot reconstructions of all nodes belonging to the cluster
# plot_centres = [i for i in range(5)]
plot_centres = np.arange(len(centers_sorted)) 
plot_centres = plot_centres[centers_sorted['kurtosis']>threshold]
reconstructions = [np.zeros(len(data_processed)) for _ in range(2)]

# plt.figure()
for ii, (node, assignment) in enumerate(
    zip(tr_obj.get_leaf_nodes(freq_threshold=freq_threshold), labels_sorted)
):
    if int(assignment) in plot_centres:
        # List reconstructions
        rec_signal = inv.inv_node(node)

        j = 0 if sf.kurtosis(trim(rec_signal, inv_keep_ratio)) > threshold else 1  # shorten name for readability
        reconstructions[j] = reconstructions[j] + rec_signal
        # psd_rec, freq = psd(rec, sampling_rate, frequency)
        # freq /= frequency
        # plt.plot(frequency, to_dB(psd_rec))
        # plt.xlim(0, fft_lim)
        # plt.title(f"frequency range: {np.divide(node.frequency_range,frequency)} (normalised by f0)")
        # plt.show()
        # plt.plot(reconstructions[0])
        # plt.plot(reconstructions[1])
        # plt.show()

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        y=reconstructions[0],
        mode="lines",
        name="higher than threshold",
    )
)
fig.add_trace(
    go.Scatter(
        y=reconstructions[1],
        mode="lines",
        name="lower than threshold",
    )
)


In [ ]:
psd_rec_higher, freq_higher = psd(reconstructions[0], sampling_rate, frequency)
psd_rec_lower, freq_lower = psd(reconstructions[1], sampling_rate, frequency)

psd_curve = psd_curve = ndi.gaussian_filter1d(psd_rec_higher, 3)

rms = np.sqrt(np.mean(psd_curve**2))
half_peak = max(psd_curve) / 2
e_peak = max(psd_curve) / np.e**2

broadband_lines = {
    "rms": rms, 
    "FWHM": half_peak, 
    "1/e<sup>2</sup> peak": e_peak
}

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=freq_higher,
        y=to_dB(psd_rec_higher),
        mode="lines",
        name="higher than threshold",
    )
)
# fig.add_trace(
#     go.Scatter(
#         x=freq_lower,
#         y=to_dB(psd_rec_lower),
#         mode="lines",
#         name="lower than threshold",
#     )
# )
fig.add_trace(
    go.Scatter(
        x=freq_higher,
        y=to_dB(psd_curve),
        name="envelope"
    )
)

for line_name, line in broadband_lines.items():
    db_line = 10 * np.log10(line)
    fig.add_trace(
        go.Scatter(
            x=freq_higher,
            y=db_line*np.ones(len(freq_higher)),
            line=dict(dash="dash"),
            name=line_name
        )
    )
fig.update_layout(
    xaxis=dict(title="f / f<sub>0</sub> [DL]", range=[0, 10]),
)

In [ ]:
import wavelet_transform.utils.statistical_features as sf

rec_high_all_nodes, rec_low_all_nodes = plot.get_reconstructions_by_threshold(
    tr_obj, feature="kurtosis", threshold=threshold, freq_threshold=freq_threshold
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    # x=freq, 
    y=tr_obj.data,
    mode='lines',
    name="original signal"
))
fig.add_trace(go.Scatter(
    # x=freq, 
    y=rec_high_all_nodes,
    mode='lines',
    name="high kurtosis reconstruction (wavelet coefficient kurtosis)"
))
fig.add_trace(go.Scatter(
    # x=freq, 
    y=rec_low_all_nodes,
    mode='lines',
    name="low kurtosis reconstruction (wavelet coefficient kurtosis)"
))
fig.show()

rec_high_kurt2, rec_low_kurt2 = plot.get_reconstructions_by_threshold(
    tr_obj, feature=sf.kurtosis, threshold=threshold, freq_threshold=freq_threshold
)
fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    # x=freq, 
    y=tr_obj.data,
    mode='lines',
    name="original signal"
))
fig2.add_trace(go.Scatter(
    # x=freq, 
    y=rec_high_kurt2,
    mode='lines',
    name="high kurtosis reconstruction (reconstruction kurtosis)"
))
fig2.add_trace(go.Scatter(
    # x=freq, 
    y=rec_low_kurt2,
    mode='lines',
    name="low kurtosis reconstruction (reconstruction kurtosis)"
))
fig2.show()


In [ ]:
# [phi, psi, x] = pywt.Wavelet('db2').wavefun(level=4)

In [ ]:
# import numpy as np
# import scipy.signal as sig
# win_func = sig.windows.blackmanharris(len(data_segments[2]), sym=False)

In [ ]:
psd_rec_higher_all_nodes, freq_higher_higher_all_nodes = psd(rec_high_all_nodes, sampling_rate, frequency)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=freq_higher_higher_all_nodes,
        y=to_dB(psd_rec_higher_all_nodes),
        mode="lines",
        name="higher than threshold_all nodes",
    )
)
fig.add_trace(
    go.Scatter(
        x=freq_higher,
        y=to_dB(psd_rec_higher),
        mode="lines",
        name="higher than threshold_ a specified cluster",
    )
)
# fig.add_trace(
#     go.Scatter(
#         x=freq_lower,
#         y=to_dB(psd_rec_lower),
#         mode="lines",
#         name="lower than threshold",
#     )
# )
fig.update_layout(
    xaxis=dict(title="f / f<sub>0</sub> [DL]", range=[0, 10]),
)

In [ ]:
from itertools import cycle, islice
lines = ['dot', 'dashdot', 'solid','dot', 'dash',  'longdash','longdashdot']
colours = ['blue','red','black','cyan','green']
linestyles = cycle(lines)
colournames = cycle(colours)
fig_thrsh = go.Figure()

# psd280 = psd_rec_higher.copy()
# freq280 = freq_higher.copy()

fig_thrsh.add_trace( 
    go.Scatter(
    x=freq_higher,
    y=to_dB(psd_rec_higher),
    mode="lines",
    name="reconstructed BB - D1",
    line=dict(color=colours[0], dash='solid', width=3)
    )
)

# fig_thrsh.add_trace( 
#     go.Scatter(
#     x=freq280,
#     y=to_dB(psd280),
#     mode="lines",
#     name="reconstructed BB - D2",
#     line=dict(color=colours[1], dash=next(linestyles), width=3)
#     )
# )

fig_thrsh.update_layout(
    height=Fig_Height,
    width=Fig_Width,
    font_size=Font_Size,
    xaxis=dict(title="f / f<sub>0</sub> [DL]",tickfont = dict(size=Tick_Size), range=[0, fft_lim]),
    yaxis=dict(title="Magnitude [dB]",tickfont = dict(size=Tick_Size), range=[-100, 50]),
    plot_bgcolor='white',
    legend=dict(
        x=0.6,
        y=1,
        traceorder='normal',
        font=dict(size=Font_Size,),
    ),
    legend_title=None,
    margin=go.layout.Margin(
        l=0, #left margin
        r=0, #right margin
        b=0, #bottom margin
        t=0  #top margin
    )
)
fig_thrsh.update_xaxes(
    mirror=True,
    ticks='outside',
    showline=True,
    linecolor='black',
    gridcolor='lightgrey'
)
fig_thrsh.update_yaxes(
    mirror=True,
    ticks='outside',
    showline=True,
    linecolor='black',
    gridcolor='lightgrey'
)
fig_thrsh.show()

if saveflag:
    import os
    FILE_PATH = os.path.abspath(".")
    RELATIVE_PATH = "./paper2024_results/"
    folder_path = os.path.abspath(os.path.join(FILE_PATH, RELATIVE_PATH))
    print("figure name: " + "GA_reconstructed_thresholded_clusters_80_280_3to6")
    save_figure(fig_thrsh, "GA_reconstructed_thresholded_clusters_80_280_3to6", folder_path)

In [ ]:
if saveflag == True:
    additional_params = {
        "raw_data": data,
        "driving frequency": frequency,
        "psd_rec_thrsh_higher_freq":freq_higher,
        "psd_rec_thrsh_higher_mag":psd_rec_higher,
        "psd_rec_thrsh_lower_freq":freq_lower,
        "psd_rec_thrsh_lower_mag":psd_rec_lower,
        "centers_sorted":centers_sorted,
        "labels_sorted":labels_sorted,
        "data_trim":data_trim
    }

    results.save_data(
        result_id, data_train, tr_obj, centers, labels, 
        sampling_rate, freq_threshold, **additional_params
    )

In [ ]:
print("Energy of the HIGH-kurtosis broadband [dB]: ",sf.log_energy(reconstructions[0]), 
      "\nEnergy of the LOW-kurtosis broadband [dB]:",sf.log_energy(reconstructions[1]))

In [ ]:
print("Energy of the HIGH-kurtosis broadband [dB] - node based calculation: ",sf.log_energy(rec_high_all_nodes), 
      "\nEnergy of the LOW-kurtosis broadband [dB] - node based calculation:",sf.log_energy(rec_low_all_nodes))

In [ ]:
print("Kurtosis of the HIGH-kurtosis broadband: ",sf.kurtosis(reconstructions[0]), 
      "\nKurtosis of the LOW-kurtosis broadband:",sf.kurtosis(reconstructions[1]))

In [ ]:
psd_rec_lower_all_nodes, freq_lower_all_nodes = psd(rec_low_all_nodes, sampling_rate, frequency)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=freq_lower_all_nodes,
        y=to_dB(psd_rec_lower_all_nodes),
        mode="lines",
        name="LOWER than threshold_all nodes",
    )
)
fig.add_trace(
    go.Scatter(
        x=freq_lower,
        y=to_dB(psd_rec_lower),
        mode="lines",
        name="LOWER than threshold_ a specified cluster",
    )
)
# fig.add_trace(
#     go.Scatter(
#         x=freq_lower,
#         y=to_dB(psd_rec_lower),
#         mode="lines",
#         name="lower than threshold",
#     )
# )
fig.update_layout(
    xaxis=dict(title="f / f<sub>0</sub> [DL]", range=[0, 10]),
)